In [1]:
# %pip install torch torchvision torchaudio pandas scikit-learn matplotlib
from torchtext.data.utils import get_tokenizer
import torch
from torch import nn
from torch.utils.data import (
    DataLoader,
    Dataset
)
from torchtext.vocab import build_vocab_from_iterator
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

df = pd.read_csv('../Dataset/Suicide_Detection.csv')
train_set, val_set = torch.utils.data.random_split(df, [0.7, 0.3])
df.head()


OSError: /home/snarkydev/Downloads/Python-movie-selfbot/venv/lib/python3.11/site-packages/torchtext/lib/libtorchtext.so: undefined symbol: _ZN5torch6detail10class_baseC2ERKSsS3_SsRKSt9type_infoS6_

In [23]:
print(df['class'].value_counts())
suicide_list = df[df['class'] == 'suicide']['text'].tolist()
non_suicide_list = df[df['class'] == 'non-suicide']['text'].tolist()

class
suicide        116037
non-suicide    116037
Name: count, dtype: int64


In [4]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        output = self.fc(output[:, -1, :])
        return output

In [7]:
from torch.utils.data import Dataset, DataLoader, random_split

class SuicideDataset(Dataset):
    def __init__(self, df, tokenizer, vocab):
        self.texts = df['text'].tolist()
        self.labels = [0 if c == 'non-suicide' else 1 for c in df['class']]
        self.tokenizer = tokenizer
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer(self.texts[idx])
        indices = [self.vocab[token] for token in tokens]
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)
    
tokenizer = get_tokenizer("basic_english")
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)
vocab = build_vocab_from_iterator(yield_tokens(df['text']), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

# Create dataset and split
dataset = SuicideDataset(df, tokenizer, vocab)
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

# DataLoader with collate_fn to pad sequences
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    text_list, label_list = [], []
    for (_text, _label) in batch:
        text_list.append(_text)
        label_list.append(_label)
    text_list = pad_sequence(text_list, batch_first=True)
    label_list = torch.tensor(label_list, dtype=torch.long)
    return text_list, label_list

vocab_size = len(vocab)
embedding_dim = 100
hidden_dim = 128
num_classes = 2
batch_size = 8
num_epochs = 10
learning_rate = 0.001

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
filename = "dectection.pth"
import pickle
import os
if not os.path.exists('./' + filename):
    # Create the model
    model = TextClassifier(vocab_size, embedding_dim, hidden_dim, num_classes).to(device)
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    # Create the data loaders
    print(f'Train loader size: {len(train_loader)}, Validation loader size: {len(valid_loader)}')
    # Iterate over the training data for the specified number of epochs

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        total_samples = 0
        
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            inputs = torch.LongTensor(inputs).to(device)
            # targets = inputs.clone()
            outputs = model(inputs).to(device)
            targets = targets.to(device)
            loss = criterion(outputs.view(-1, num_classes), targets.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(inputs)
            total_samples += len(inputs)
            

        # Evaluate on the validation set after every epoch
        model.eval()
        total_val_loss = 0.0
        total_val_samples = 0
        with torch.no_grad():
            for inputs, targets in valid_loader:
                inputs = torch.LongTensor(inputs).to(device)
                # targets = inputs.clone()
                outputs = model(inputs).to(device)
                targets = targets.to(device)
                val_loss = criterion(outputs.view(-1, num_classes), targets.view(-1))

                total_val_loss += val_loss.item() * len(inputs)
                total_val_samples += len(inputs)

        avg_loss = total_loss / total_samples
        avg_val_loss = total_val_loss / total_val_samples

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    torch.save(model.state_dict(), filename)
    print("Model saved to " + filename)
else:
    with open('vocab.pkl', 'wb') as f:
        vocab = pickle.dump(vocab, f)
    model = TextClassifier(vocab_size, embedding_dim, hidden_dim, num_classes)
    model.load_state_dict(torch.load(filename))
    model.to(device)


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/home/snarkydev/Downloads/Python-movie-selfbot/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train loader size: 20307, Validation loader size: 8703
Epoch 1/10, Train Loss: 0.2597, Val Loss: 0.1616
Epoch 2/10, Train Loss: 0.1346, Val Loss: 0.1441
Epoch 3/10, Train Loss: 0.1038, Val Loss: 0.1432
Epoch 4/10, Train Loss: 0.0812, Val Loss: 0.1442
Epoch 5/10, Train Loss: 0.0636, Val Loss: 0.1592
Epoch 6/10, Train Loss: 0.0500, Val Loss: 0.1648
Epoch 7/10, Train Loss: 0.0410, Val Loss: 0.1723
Epoch 8/10, Train Loss: 0.0330, Val Loss: 0.1904
Epoch 9/10, Train Loss: 0.0297, Val Loss: 0.2009
Epoch 10/10, Train Loss: 0.0259, Val Loss: 0.2120
Model saved to dectection.pth


In [27]:
def predict_text(text, model, tokenizer, vocab, device):
    model.eval()
    tokens = tokenizer(text)
    indices = [vocab[token] for token in tokens]
    input_tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)  # Add batch dimension
    with torch.no_grad():
        output = model(input_tensor)
        predicted_class = output.argmax(dim=1).item()
    return predicted_class

# Example 
# neg:
# test_text = "I feel so hopeless and alone, I don't know how to go on anymore."
# pos:
test_text = "I am so happy and grateful for my life :)"
result = predict_text(test_text, model, tokenizer, vocab, device)
print(f"Model output for test_text: {result}")
if result == 1:
    print("Prediction: suicide")
else:
    print("Prediction: non-suicide")

Model output for test_text: 0
Prediction: non-suicide
